# Role B (diacritic corrector): first session

Needs the Colab secret `GITHUB_TOKEN` with **Notebook access** switched on for this notebook.

This notebook only looks at the corpus and measures how well it matches the OCR labels. Your own work goes into `src/corrector/` and into notebooks named `corrector_*`. Colab cannot push to GitHub: download your notebook (File > Download .ipynb), put it in the `notebooks/` folder of your clone and push from your laptop.

**First:** choose File > Save a copy in Drive and work in the copy, so the notebook in the shared folder stays clean for everybody. In the copy, switch on Notebook access for `GITHUB_TOKEN` again.


## 1. Connect to Drive and GitHub

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

BRANCH = "main"  # use your own branch (a/ocr, b/corrector, c/eval) to test code you have pushed

token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/repo
!git clone -q https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
!git -C /content/repo checkout -q {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "/content/repo")

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"


## 2. Load the corpus

In [ ]:
from collections import Counter, defaultdict

from src.common.text_utils import strip_diacritics

CORPUS = f"{DATA_DIR}/corpus/viwiki_sentences.txt"
with open(CORPUS, encoding="utf-8") as handle:
    sentences = [line.rstrip("\n") for line in handle][:200000]
print("sentences:", len(sentences))

def clean(token):
    return token.lower().strip(".,;:!?()'\"-%/")

tokens = [clean(tok) for sentence in sentences for tok in sentence.split(" ")]
vocab = Counter(tok for tok in tokens if tok)
print("syllable tokens:", len(tokens), "distinct:", len(vocab))


## 3. See what the noise looks like (the same text without diacritics)

In [ ]:
for sentence in sentences[:5]:
    print(sentence)
    print(strip_diacritics(sentence))
    print()


## 4. Which plain syllables are the most ambiguous?

In [ ]:
forms = defaultdict(Counter)
for syllable, count in vocab.items():
    forms[strip_diacritics(syllable)][syllable] += count

ranked = sorted(forms.items(), key=lambda item: -len(item[1]))[:15]
for plain, counter in ranked:
    print("%-6s %2d forms  %s" % (plain, len(counter), counter.most_common(3)))


## 5. Does this corpus match the OCR labels?

Share of syllables in the UIT-HWDB **training** labels that also appear in the corpus. Use only the training labels for this kind of comparison, never the test labels.


In [ ]:
import os

index_path = f"{OUT_DIR}/dataset_index.jsonl"
if os.path.exists(index_path):
    from src.common.index_io import read_index, read_splits, select_split
    train = select_split(read_index(index_path), read_splits(f"{OUT_DIR}/splits.json"), "train")
    label_tokens = [clean(tok) for rec in train for tok in rec["gt_text"].split(" ")]
    label_tokens = [tok for tok in label_tokens if tok]
    covered = sum(1 for tok in label_tokens if tok in vocab)
    print("label syllables covered by the corpus: %.1f%%" % (100.0 * covered / len(label_tokens)))
else:
    print("dataset_index.jsonl not found yet")


## What to do next

1. Write your own noise generator (remove diacritics, swap tone marks, swap vowel shape marks, each with a probability) and make training pairs (noisy sentence, clean sentence).
2. Frequency-table baseline, then bigram + Viterbi by hand. Measure syllable accuracy overall and on ambiguous syllables.
3. A first `correct()` that follows the contract (`{"text": str, "conf": [float]}`) as soon as the baseline works, so role C can integrate it early.
4. Save checkpoints to `CKPT_DIR/corrector` after every epoch once you train a neural model.
